In [ ]:
import xarray as xr
import rioxarray
import numpy as np
import glob
import re
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.append('..')
from utils import retrieve_url

## Retrieve Files

In [ ]:
bands = [616, 620]
base_filename = "hrrr.t00z.wrfprsf00"
base_url = "https://demo.openwfm.org/web/data/fmda//tif/20240101/"

In [ ]:
for band in bands:
    filename = f"{base_filename}.{band}.tif"
    retrieve_url(
        f"{base_url}/{filename}",
        dest_path = f"{filename}"
    )

## Handle Single Datasets

In [ ]:
ds1 = xr.open_dataset(f"{base_filename}.616.tif")
ds2 = xr.open_dataset(f"{base_filename}.620.tif")

In [ ]:
type(ds1)

In [ ]:
ds1.band_data.shape

In [ ]:
print(ds1.band_data.dims)
print(ds1.band_data.coords)

In [ ]:
ds2.band_data.shape

In [ ]:
temp = ds1.band_data
rh = ds2.band_data

In [ ]:
type(temp)

In [ ]:
Ed = 0.924*rh**0.679 + 0.000499*np.exp(0.1*rh) + 0.18*(21.1 + 273.15 - temp)*(1 - np.exp(-0.115*rh))
Ew = 0.618*rh**0.753 + 0.000454*np.exp(0.1*rh) + 0.18*(21.1 + 273.15 - temp)*(1 - np.exp(-0.115*rh)) 

In [ ]:
type(Ed)

In [ ]:
Ed.dims

In [ ]:
plt.imshow(Ed.isel(band=0))

In [ ]:
type(Ed)

## Open Multiple Datasets

In [ ]:
file_list = glob.glob(f"{base_filename}*.tif")
print(file_list)

In [ ]:
data = xr.open_mfdataset(file_list, concat_dim="band", combine="nested")

In [ ]:
data

In [ ]:
data.dims

In [ ]:
band_df_hrrr = pd.DataFrame({
    'Band': [616, 620, 624, 628, 629, 661, 561, 612, 643],
    'hrrr_name': ['TMP', 'RH', "WIND", 'PRATE', 'APCP',
                  'DSWRF', 'SOILW', 'CNWAT', 'GFLUX'],
    'dict_name': ["temp", "rh", "wind", "rain", "precip_accum",
                 "solar", "soilm", "canopyw", "groundflux"],
    'descr': ['2m Temperature [K]', 
              '2m Relative Humidity [%]', 
              '10m Wind Speed [m/s]'
              'surface Precip. Rate [kg/m^2/s]',
              'surface Total Precipitation [kg/m^2]',
              'surface Downward Short-Wave Radiation Flux [W/m^2]',
              'surface Total Precipitation [kg/m^2]',
              '0.0m below ground Volumetric Soil Moisture Content [Fraction]',
              'Plant Canopy Surface Water [kg/m^2]',
              'surface Ground Heat Flux [W/m^2]']
})

In [ ]:
file_list

In [ ]:
# Use regex to extract the number before the final period and extension
numbers = [re.search(r'\.(\d+)\.[^.]+$', filename).group(1) for filename in file_list]

# Convert the extracted numbers to integers, if desired
numbers = [int(num) for num in numbers]

print(numbers)  # Output: [616, 620]

In [ ]:
def band_from_hrrrname(filename):
    # Extract the number before the final extension for each file
    numbers = [int(re.search(r'\.(\d+)\.[^.]+$', filename).group(1)) for filename in file_list]
    return numbers 

def get_dict_names(file_list, band_df):
    # Get the bands from file names
    bands = band_from_hrrrname(file_list)
    # Find matching dict_name values in the dataframe
    dict_names = band_df.loc[band_df['Band'].isin(bands), 'dict_name'].tolist()
    return dict_names

In [ ]:
band_from_hrrrname(file_list)

In [ ]:
get_dict_names(file_list, band_df_hrrr)

In [ ]:
type(data)

In [ ]:
def calc_eqs(data, file_list):
    # Get band numbers from file_list
    band_numbers = band_from_hrrrname(file_list)

    # Define the specific band numbers for temp and rh based on known mapping
    temp_band_number = 616  # Band number for temp
    rh_band_number = 620    # Band number for rh

    # Find the indices of temp and rh in the data's band dimension
    try:
        temp_index = band_numbers.index(temp_band_number)
        rh_index = band_numbers.index(rh_band_number)
    except ValueError:
        raise ValueError("Required bands for temp or rh not found in file_list.")

    # Calculate Ed based on temp and rh
    Ed = (
        0.924 * data.isel(band=rh_index)**0.679
        + 0.000499 * np.exp(0.1 * data.isel(band=rh_index))
        + 0.18 * (21.1 + 273.15 - data.isel(band=temp_index)) * (1 - np.exp(-0.115 * data.isel(band=rh_index)))
    )
    Ew = 0.618 * data.isel(band=rh_index)**0.753 + 0.000454 * np.exp(0.1 * data.isel(band=rh_index)) + 0.18 * (21.1 + 273.15 - data.isel(band=temp_index)) * (1 - np.exp(-0.115 * data.isel(band=rh_index)))


    # Modify the Dataset in place by adding Ed
    data['Ed'] = Ed
    data['Ew'] = Ew
    

In [ ]:
data.dims

In [ ]:
calc_eqs(data, file_list)

In [ ]:
band_numbers = band_from_hrrrname(file_list)
rh_band_number = 620
temp_band_number = 616  # Band number for temp
rh_index = band_numbers.index(rh_band_number)
temp_index = band_numbers.index(temp_band_number)
data.isel(band=rh_index)

In [ ]:
type(data.isel(band=rh_index))

In [ ]:
Ed = (
    0.924 * data.isel(band=rh_index)**0.679
    + 0.000499 * np.exp(0.1 * data.isel(band=rh_index))
    + 0.18 * (21.1 + 273.15 - data.isel(band=temp_index)) * (1 - np.exp(-0.115 * data.isel(band=rh_index)))
)
Ew = 0.618 * data.isel(band=rh_index)**0.753 + 0.000454 * np.exp(0.1 * data.isel(band=rh_index)) + 0.18 * (21.1 + 273.15 - data.isel(band=temp_index)) * (1 - np.exp(-0.115 * data.isel(band=rh_index)))

In [ ]:
data = xr.concat([data, Ed, Ew], dim="band")

In [ ]:
data.dims

In [ ]:
Ed = data.isel(band=2)['band_data']

# plt.imshow(Ed, origin='upper', aspect='auto')
plt.imshow(Ed)
plt.colorbar(label="Data Value")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Band 2 Plot")
plt.show()

In [ ]:
Ed = data.isel(band=2)['band_data']

In [ ]:
type(Ed)